# Day 3: Deep Q-Networks (DQN)

**Reinforcement Learning: From Theory to Practice**

---

## Overview

Tabular methods cannot scale to large or continuous state spaces. Today we combine Q-Learning with **neural network function approximation**:

1. **Function approximation** concepts
2. **DQN** -- Deep Q-Network with PyTorch
3. **Experience replay** buffer
4. **Target network** for stability
5. **Training on CartPole-v1** (gymnasium)
6. **Double DQN** variant to reduce overestimation bias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import gymnasium as gym

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 1. Function Approximation -- Why?

In tabular RL, we store $Q(s, a)$ for every state-action pair. For CartPole, the state is a 4D continuous vector $(x, \dot{x}, \theta, \dot{\theta})$. With discretization, even a coarse grid would have thousands of cells. Deep RL uses a neural network $Q_\theta(s, a)$ to **generalize** across similar states.

### Challenges of Naive Q-Learning with Neural Nets

1. **Correlated samples** -- consecutive transitions are highly correlated, destabilizing SGD.
2. **Non-stationary targets** -- the target $r + \gamma \max_a Q_\theta(s', a)$ changes as we update $\theta$.

DQN (Mnih et al., 2015) addresses both with **experience replay** and a **target network**.

## 2. Experience Replay Buffer

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    """
    Fixed-size circular buffer for experience replay.
    
    Stores transitions (s, a, r, s', done) and provides
    uniform random sampling for mini-batch training.
    """
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """Sample a random mini-batch of transitions."""
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).unsqueeze(1).to(device)
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)


# Quick test
buf = ReplayBuffer(capacity=100)
for i in range(50):
    buf.push(np.random.randn(4), np.random.randint(2), np.random.randn(),
             np.random.randn(4), False)

states, actions, rewards, next_states, dones = buf.sample(8)
print(f'Batch shapes: states={states.shape}, actions={actions.shape}, '
      f'rewards={rewards.shape}, next_states={next_states.shape}, dones={dones.shape}')

## 3. Q-Network Architecture

In [ ]:
class QNetwork(nn.Module):
    """
    Simple feedforward neural network for Q-value approximation.
    
    Input: state vector (dim = state_dim)
    Output: Q-values for each action (dim = n_actions)
    """
    
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
    
    def forward(self, x):
        return self.net(x)


# Test the network
state_dim = 4  # CartPole
n_actions = 2

q_net = QNetwork(state_dim, n_actions).to(device)
test_state = torch.randn(1, state_dim).to(device)
q_values = q_net(test_state)
print(f'Q-Network output for random state: {q_values.detach().cpu().numpy()}')
print(f'Total parameters: {sum(p.numel() for p in q_net.parameters()):,}')

## 4. DQN Agent

The DQN agent combines:
- **Online network** $Q_\theta$ -- updated every step
- **Target network** $Q_{\theta^-}$ -- updated periodically (hard copy or soft update)
- **Experience replay** -- break correlations
- **Epsilon-greedy** exploration with decay

In [ ]:
class DQNAgent:
    """
    Deep Q-Network agent with experience replay and target network.
    """
    
    def __init__(self, state_dim, n_actions, hidden_dim=128,
                 lr=1e-3, gamma=0.99, buffer_size=10000,
                 batch_size=64, target_update_freq=100,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=500,
                 double_dqn=False):
        
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.double_dqn = double_dqn
        
        # Epsilon schedule
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Networks
        self.q_network = QNetwork(state_dim, n_actions, hidden_dim).to(device)
        self.target_network = QNetwork(state_dim, n_actions, hidden_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()  # target network is not trained directly
        
        # Optimizer
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(capacity=buffer_size)
        
        # Step counter
        self.steps_done = 0
    
    def get_epsilon(self):
        """Exponentially decaying epsilon."""
        eps = self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
              np.exp(-self.steps_done / self.epsilon_decay)
        return eps
    
    def select_action(self, state):
        """Epsilon-greedy action selection."""
        eps = self.get_epsilon()
        self.steps_done += 1
        
        if np.random.random() < eps:
            return np.random.randint(self.n_actions)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_vals = self.q_network(state_t)
                return q_vals.argmax(dim=1).item()
    
    def update(self):
        """Perform one gradient step on the Q-network."""
        if len(self.replay_buffer) < self.batch_size:
            return None
        
        states, actions, rewards, next_states, dones = \
            self.replay_buffer.sample(self.batch_size)
        
        # Current Q-values for chosen actions
        current_q = self.q_network(states).gather(1, actions)
        
        # Target Q-values
        with torch.no_grad():
            if self.double_dqn:
                # Double DQN: use online net to SELECT actions,
                # target net to EVALUATE them
                best_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
                next_q = self.target_network(next_states).gather(1, best_actions)
            else:
                # Standard DQN: target net for both selection and evaluation
                next_q = self.target_network(next_states).max(dim=1, keepdim=True)[0]
            
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Loss
        loss = F.smooth_l1_loss(current_q, target_q)  # Huber loss
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        return loss.item()
    
    def update_target(self):
        """Hard update: copy online network weights to target network."""
        self.target_network.load_state_dict(self.q_network.state_dict())


print('DQNAgent class defined.')

## 5. Training on CartPole-v1

CartPole: balance a pole on a cart by pushing left or right.
- **State**: $(x, \dot{x}, \theta, \dot{\theta})$ -- 4 continuous values
- **Actions**: 0 (push left), 1 (push right)
- **Reward**: +1 per time step the pole stays upright
- **Solved**: average reward >= 475 over 100 consecutive episodes

In [ ]:
def train_dqn(env_name='CartPole-v1', n_episodes=300, double_dqn=False,
              seed=42, verbose=True):
    """
    Train a DQN agent on a gymnasium environment.
    
    Returns
    -------
    agent : trained DQNAgent
    episode_rewards : list of total rewards per episode
    losses : list of training losses
    """
    env = gym.make(env_name)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    
    state_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    
    agent = DQNAgent(
        state_dim=state_dim,
        n_actions=n_actions,
        hidden_dim=128,
        lr=1e-3,
        gamma=0.99,
        buffer_size=10000,
        batch_size=64,
        target_update_freq=100,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=500,
        double_dqn=double_dqn
    )
    
    episode_rewards = []
    losses = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        total_reward = 0
        ep_losses = []
        
        done = False
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.replay_buffer.push(state, action, reward, next_state, float(done))
            
            loss = agent.update()
            if loss is not None:
                ep_losses.append(loss)
            
            # Update target network
            if agent.steps_done % agent.target_update_freq == 0:
                agent.update_target()
            
            state = next_state
            total_reward += reward
        
        episode_rewards.append(total_reward)
        if ep_losses:
            losses.append(np.mean(ep_losses))
        
        if verbose and (ep + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            eps = agent.get_epsilon()
            print(f'Episode {ep+1:4d} | Avg Reward (50): {avg:6.1f} | '
                  f'Epsilon: {eps:.3f} | Buffer: {len(agent.replay_buffer)}')
    
    env.close()
    return agent, episode_rewards, losses


# Train standard DQN
print('=== Training Standard DQN ===')
agent_dqn, rewards_dqn, losses_dqn = train_dqn(double_dqn=False, n_episodes=300)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rewards
ax = axes[0]
ax.plot(rewards_dqn, alpha=0.3, color='steelblue')
window = 20
if len(rewards_dqn) >= window:
    smoothed = np.convolve(rewards_dqn, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards_dqn)), smoothed,
            linewidth=2, color='darkblue', label=f'{window}-ep avg')
ax.axhline(y=475, color='green', linestyle='--', label='Solved threshold (475)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('DQN Training -- CartPole-v1')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1]
ax.plot(losses_dqn, alpha=0.5, color='coral')
ax.set_xlabel('Episode')
ax.set_ylabel('Mean Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Double DQN

Standard DQN tends to **overestimate** Q-values because it uses the same network to both select and evaluate actions in the target:

$$y = r + \gamma \max_a Q_{\theta^-}(s', a)$$

**Double DQN** (van Hasselt et al., 2016) decouples selection and evaluation:

$$y = r + \gamma Q_{\theta^-}\big(s',\; \arg\max_a Q_\theta(s', a)\big)$$

The online network **selects** the best action; the target network **evaluates** it.

In [ ]:
# Train Double DQN
print('=== Training Double DQN ===')
agent_ddqn, rewards_ddqn, losses_ddqn = train_dqn(double_dqn=True, n_episodes=300)

In [ ]:
# Compare DQN vs Double DQN
fig, ax = plt.subplots(figsize=(10, 5))

window = 20
for rewards, label, color in [(rewards_dqn, 'DQN', '#2196F3'),
                               (rewards_ddqn, 'Double DQN', '#FF5722')]:
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed,
                linewidth=2, color=color, label=label)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.5, label='Solved')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('DQN vs Double DQN on CartPole-v1')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluating the Trained Agent

In [ ]:
def evaluate_agent(agent, env_name='CartPole-v1', n_episodes=20, seed=0):
    """Run the agent greedily (no exploration) and report performance."""
    env = gym.make(env_name)
    rewards = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        total_reward = 0
        done = False
        
        while not done:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = agent.q_network(state_t).argmax(dim=1).item()
            
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        
        rewards.append(total_reward)
    
    env.close()
    return rewards


eval_rewards_dqn = evaluate_agent(agent_dqn)
eval_rewards_ddqn = evaluate_agent(agent_ddqn)

print(f'DQN Evaluation   : mean={np.mean(eval_rewards_dqn):.1f}, '
      f'std={np.std(eval_rewards_dqn):.1f}, '
      f'min={np.min(eval_rewards_dqn):.0f}, max={np.max(eval_rewards_dqn):.0f}')
print(f'Double DQN Eval  : mean={np.mean(eval_rewards_ddqn):.1f}, '
      f'std={np.std(eval_rewards_ddqn):.1f}, '
      f'min={np.min(eval_rewards_ddqn):.0f}, max={np.max(eval_rewards_ddqn):.0f}')

## 8. Visualizing Q-Values

Let us see how the Q-network's outputs vary as we change one state dimension (the pole angle $\theta$) while keeping others fixed.

In [ ]:
# Sweep over pole angle
angles = np.linspace(-0.3, 0.3, 200)
q_left = []
q_right = []

for theta in angles:
    state = np.array([0.0, 0.0, theta, 0.0])  # x=0, xdot=0, theta, thetadot=0
    with torch.no_grad():
        q = agent_dqn.q_network(torch.FloatTensor(state).unsqueeze(0).to(device))
    q_left.append(q[0, 0].item())
    q_right.append(q[0, 1].item())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.degrees(angles), q_left, label='Q(s, left)', linewidth=2, color='#2196F3')
ax.plot(np.degrees(angles), q_right, label='Q(s, right)', linewidth=2, color='#F44336')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Pole Angle (degrees)')
ax.set_ylabel('Q-value')
ax.set_title('Learned Q-values vs Pole Angle (other dims = 0)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('When the pole tilts right (positive angle), Q(right) > Q(left) --')
print('the agent has learned to push in the direction the pole is falling.')

## Summary

| Component | Purpose |
|-----------|--------|
| **Q-Network** | Approximate $Q(s,a)$ with a neural network |
| **Experience Replay** | Break temporal correlations, improve sample efficiency |
| **Target Network** | Stabilize training by fixing targets periodically |
| **Double DQN** | Reduce overestimation by decoupling action selection/evaluation |

### Key Takeaways

- DQN was the first deep RL algorithm to achieve human-level performance on Atari games.
- The replay buffer and target network are critical for stable training.
- Double DQN is a simple modification that often improves performance.
- Other improvements exist: Dueling DQN, Prioritized Experience Replay, Noisy Nets, etc.

**Next:** Day 4 -- Policy Gradient methods (REINFORCE, Actor-Critic, PPO)